# Vesuvius V3 Inference with Hysteresis Thresholding

## Key Features
- **Seeded Hysteresis Thresholding**: Two-threshold approach where confident regions (high threshold) propagate through uncertain regions (low threshold)
- **Binary Propagation**: Uses scipy.ndimage.binary_propagation for connectivity-aware expansion
- **Anisotropic Morphological Closing**: Optional closing with ellipsoid structuring element
- **Dust Removal**: Removes small disconnected components

**Model**: ResEncUNet3D with SafeInstanceNorm3d (Val Dice 0.62)

In [ ]:
!pip install imagecodecs connected-components-3d tifffile -q

In [ ]:
import os
import sys
import gc
import zipfile
import warnings
from pathlib import Path
from typing import Tuple, List

import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from scipy import ndimage as ndi
from skimage.morphology import remove_small_objects, ball, closing

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False
    print("cc3d not found, using scipy for connected components")

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    TEST_IMAGES_DIR = Path("/kaggle/input/vesuvius-challenge-surface-detection/test_images")
    TEST_CSV = Path("/kaggle/input/vesuvius-challenge-surface-detection/test.csv")
    
    # UPDATE THIS PATH to your V3 trained model checkpoint
    CHECKPOINT_PATH = Path("/kaggle/input/vesuvius-v3-checkpoints/best_model.pth")
    
    OUTPUT_DIR = Path("/kaggle/working/submission_masks")
    SUBMISSION_ZIP = Path("/kaggle/working/submission.zip")
    
    # Architecture - MUST match V3 training with SafeInstanceNorm3d
    PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)
    FEATURES: List[int] = [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = [1, 3, 4, 6, 6, 6]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # Inference settings
    OVERLAP: float = 0.5
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    
    # ==========================================================================
    # HYSTERESIS THRESHOLDING PARAMETERS
    # ==========================================================================
    # V3 model probability distribution:
    #   - Mean: 0.041, Std: 0.059
    #   - At 0.03: 36.7% (too much)
    #   - At 0.15: 0.95% (seeds)
    #   - Target: ~10% FG
    
    # High threshold: Seeds (confident regions)
    THRESHOLD_HIGH: float = 0.12  # ~1-2% seeds
    
    # Low threshold: Expansion boundary
    # Need to find threshold that gives ~10-12% before hysteresis
    # Linear interpolation: 0.03->36.7%, need ~10%, so try ~0.06-0.07
    THRESHOLD_LOW: float = 0.065  # Target ~10-12% weak regions
    
    # Morphological closing (helps connect fragmented vessels)
    USE_CLOSING: bool = True
    CLOSING_RADIUS: int = 2  # Reduced since we have better threshold now
    
    # Dust removal
    MIN_COMPONENT_SIZE: int = 50  # Standard dust removal

cfg = Config()
cfg.OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"Device: {cfg.DEVICE}")
if cfg.DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} (CUDA {props.major}.{props.minor})")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

print(f"\nHysteresis thresholds: HIGH={cfg.THRESHOLD_HIGH}, LOW={cfg.THRESHOLD_LOW}")
print(f"Morphological closing: {cfg.USE_CLOSING} (radius={cfg.CLOSING_RADIUS})")
print(f"Min component size: {cfg.MIN_COMPONENT_SIZE}")

In [ ]:
# =============================================================================
# MODEL - SafeInstanceNorm3d (MUST match V3 training)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    """
    InstanceNorm3d with safeguards against division by very small variances.
    CRITICAL: Must match the V3 training notebook exactly!
    """
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super(SafeInstanceNorm3d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        
        if self.affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)
    
    def forward(self, x):
        mean = x.mean(dim=[2, 3, 4], keepdim=True)
        var = x.var(dim=[2, 3, 4], keepdim=True, unbiased=False)
        var_safe = torch.clamp(var, min=self.eps)
        x_norm = (x - mean) / torch.sqrt(var_safe + self.eps)
        
        if self.affine:
            x_norm = self.weight.view(1, -1, 1, 1, 1) * x_norm + self.bias.view(1, -1, 1, 1, 1)
        
        return x_norm


class ConvBlock(nn.Module):
    """3D convolution + SafeInstanceNorm3d + LeakyReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),
            SafeInstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )

    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)

    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    """V3 ResEncUNet with SafeInstanceNorm3d."""

    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()

        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]

        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)

        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()

        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)

            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())

            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2, bias=True))

        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()

        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2, bias=True))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))

        self.final = nn.Conv3d(features[0], out_ch, 1, bias=True)

        # Deep supervision heads (needed for weight loading)
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(4, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1, bias=True))

    def forward(self, x):
        enc_features = []

        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)

        enc_features = enc_features[::-1]
        x = enc_features[0]

        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)

        return self.final(x)

In [ ]:
# =============================================================================
# HYSTERESIS THRESHOLDING POST-PROCESSING
# =============================================================================

def create_anisotropic_structuring_element(radius_xy, radius_z=None):
    """
    Create an anisotropic ellipsoid structuring element.
    
    For volumes with different resolutions in different axes,
    this creates an ellipsoid that accounts for the anisotropy.
    
    Args:
        radius_xy: Radius in X and Y dimensions
        radius_z: Radius in Z dimension (if None, uses radius_xy)
    
    Returns:
        3D boolean structuring element
    """
    if radius_z is None:
        radius_z = radius_xy
    
    # Create coordinate grids
    z = np.arange(-radius_z, radius_z + 1)
    y = np.arange(-radius_xy, radius_xy + 1)
    x = np.arange(-radius_xy, radius_xy + 1)
    
    Z, Y, X = np.meshgrid(z, y, x, indexing='ij')
    
    # Ellipsoid equation: (x/rx)^2 + (y/ry)^2 + (z/rz)^2 <= 1
    ellipsoid = ((X / max(radius_xy, 1))**2 + 
                 (Y / max(radius_xy, 1))**2 + 
                 (Z / max(radius_z, 1))**2) <= 1
    
    return ellipsoid


def hysteresis_threshold_3d(prob_map, threshold_high, threshold_low):
    """
    Apply seeded hysteresis thresholding to a 3D probability map.
    
    This is a two-threshold segmentation method:
    1. Strong threshold creates "seed" regions (high confidence)
    2. Weak threshold defines the expansion boundary
    3. Seeds grow to fill connected weak regions
    
    This approach:
    - Reduces false positives (isolated low-confidence regions are removed)
    - Maintains connectivity (weak regions connected to strong ones are kept)
    - Preserves thin structures that have at least some high-confidence voxels
    
    Args:
        prob_map: 3D probability array (values 0-1)
        threshold_high: High threshold for seeds (e.g., 0.80)
        threshold_low: Low threshold for expansion (e.g., 0.35)
    
    Returns:
        Binary mask after hysteresis thresholding
    """
    # Create seed regions (high confidence)
    seeds = prob_map >= threshold_high
    
    # Create expansion boundary (weak regions)
    weak = prob_map >= threshold_low
    
    # Use binary propagation to grow seeds within weak regions
    # This uses 26-connectivity (full 3D neighborhood)
    struct = ndi.generate_binary_structure(3, 3)  # 26-connectivity
    
    # Propagate seeds through the weak mask
    # Only voxels in 'weak' that are connected to 'seeds' will be included
    result = ndi.binary_propagation(seeds, structure=struct, mask=weak)
    
    return result.astype(bool)


def apply_morphological_closing(mask, radius):
    """
    Apply morphological closing to connect nearby structures.
    
    Closing = dilation followed by erosion.
    This fills small gaps and holes while preserving the overall shape.
    
    Args:
        mask: Binary 3D mask
        radius: Radius of the structuring element
    
    Returns:
        Closed binary mask
    """
    if radius <= 0:
        return mask
    
    # Use ball structuring element (isotropic)
    selem = ball(radius)
    
    # Apply closing
    closed = closing(mask, selem)
    
    return closed


def remove_small_components(mask, min_size, connectivity=26):
    """
    Remove small connected components (dust removal).
    
    Args:
        mask: Binary 3D mask
        min_size: Minimum component size to keep
        connectivity: 6 (face), 18 (edge), or 26 (vertex) connectivity
    
    Returns:
        Cleaned binary mask
    """
    if min_size <= 0:
        return mask
    
    # Use skimage's remove_small_objects
    # Convert connectivity to the right format
    if connectivity == 26:
        conn = 3  # Full connectivity in 3D
    elif connectivity == 18:
        conn = 2
    else:
        conn = 1  # 6-connectivity
    
    cleaned = remove_small_objects(mask.astype(bool), min_size=min_size, connectivity=conn)
    
    return cleaned


def hysteresis_postprocess(prob_map, cfg):
    """
    Full post-processing pipeline with hysteresis thresholding.
    
    Pipeline:
    1. Hysteresis thresholding (seed + propagate)
    2. Morphological closing (optional - connect nearby structures)
    3. Dust removal (remove small components)
    
    Args:
        prob_map: 3D probability array (values 0-1)
        cfg: Configuration object with thresholds and parameters
    
    Returns:
        Final binary mask (uint8)
    """
    print(f"  Probability range: [{prob_map.min():.4f}, {prob_map.max():.4f}]")
    print(f"  Prob mean: {prob_map.mean():.4f}, std: {prob_map.std():.4f}")
    
    # Statistics before hysteresis
    seeds_pct = 100 * (prob_map >= cfg.THRESHOLD_HIGH).mean()
    weak_pct = 100 * (prob_map >= cfg.THRESHOLD_LOW).mean()
    print(f"  Seeds (>= {cfg.THRESHOLD_HIGH}): {seeds_pct:.3f}%")
    print(f"  Weak regions (>= {cfg.THRESHOLD_LOW}): {weak_pct:.3f}%")
    
    # Step 1: Hysteresis thresholding
    print(f"  Applying hysteresis thresholding...")
    mask = hysteresis_threshold_3d(
        prob_map,
        threshold_high=cfg.THRESHOLD_HIGH,
        threshold_low=cfg.THRESHOLD_LOW
    )
    hysteresis_pct = 100 * mask.mean()
    print(f"  After hysteresis: {hysteresis_pct:.3f}%")
    
    # Step 2: Morphological closing (optional)
    if cfg.USE_CLOSING and cfg.CLOSING_RADIUS > 0:
        print(f"  Applying morphological closing (radius={cfg.CLOSING_RADIUS})...")
        mask = apply_morphological_closing(mask, cfg.CLOSING_RADIUS)
        closing_pct = 100 * mask.mean()
        print(f"  After closing: {closing_pct:.3f}%")
    
    # Step 3: Remove small components (dust)
    if cfg.MIN_COMPONENT_SIZE > 0:
        print(f"  Removing components < {cfg.MIN_COMPONENT_SIZE} voxels...")
        mask = remove_small_components(mask, cfg.MIN_COMPONENT_SIZE)
        final_pct = 100 * mask.mean()
        print(f"  Final FG%: {final_pct:.3f}%")
    
    return mask.astype(np.uint8)

In [ ]:
# =============================================================================
# NORMALIZATION & INFERENCE
# =============================================================================

def normalize_volume(volume: np.ndarray) -> np.ndarray:
    """Simple z-score normalization (matches V3 training)."""
    return (volume.astype(np.float32) - volume.mean()) / (volume.std() + 1e-8)


@torch.no_grad()
def sliding_window_inference(model, volume, patch_size=(128, 128, 128), overlap=0.5, device="cuda", use_amp=True):
    """Sliding window inference with Gaussian blending."""
    model.eval()
    model.to(device)
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd*(1-overlap)), int(ph*(1-overlap)), int(pw*(1-overlap))
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Gaussian blending weights
    gz = np.exp(-0.5*((np.arange(pd)-pd/2)/(pd*0.125))**2)
    gy = np.exp(-0.5*((np.arange(ph)-ph/2)/(ph*0.125))**2)
    gx = np.exp(-0.5*((np.arange(pw)-pw/2)/(pw*0.125))**2)
    gauss = (gz[:,None,None] * gy[None,:,None] * gx[None,None,:]).astype(np.float32)
    
    # Generate patch positions
    z_pos = list(range(0, max(1, D-pd+1), sd))
    if D > pd and (D-pd) not in z_pos: z_pos.append(D-pd)
    y_pos = list(range(0, max(1, H-ph+1), sh))
    if H > ph and (H-ph) not in y_pos: y_pos.append(H-ph)
    x_pos = list(range(0, max(1, W-pw+1), sw))
    if W > pw and (W-pw) not in x_pos: x_pos.append(W-pw)
    
    # Process patches
    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = volume[z:z+pd, y:y+ph, x:x+pw].astype(np.float32)
                actual_shape = patch.shape
                
                # Pad if at volume edges
                if patch.shape != (pd, ph, pw):
                    pad = [(0, max(0, pd-patch.shape[0])), 
                           (0, max(0, ph-patch.shape[1])), 
                           (0, max(0, pw-patch.shape[2]))]
                    patch = np.pad(patch, pad, mode='reflect')
                
                # Normalize
                patch = normalize_volume(patch)
                inp = torch.from_numpy(patch[None, None]).to(device)
                
                # Inference
                with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                    out = torch.sigmoid(model(inp)).squeeze().cpu().numpy()
                
                # Crop to actual size and accumulate
                out = out[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                weight = gauss[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                pred_sum[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += out * weight
                count[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += weight
    
    return pred_sum / np.maximum(count, 1e-8)

In [ ]:
# =============================================================================
# MAIN INFERENCE PIPELINE
# =============================================================================

def main():
    # Load test metadata
    test_df = pd.read_csv(cfg.TEST_CSV)
    print(f"Found {len(test_df)} test volumes")
    
    # Initialize model
    print("\nInitializing model with SafeInstanceNorm3d...")
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION
    )
    
    # Load weights
    print(f"Loading weights from: {cfg.CHECKPOINT_PATH}")
    if not cfg.CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Checkpoint not found: {cfg.CHECKPOINT_PATH}")
    
    checkpoint = torch.load(cfg.CHECKPOINT_PATH, map_location=cfg.DEVICE, weights_only=False)
    
    # Handle checkpoint format
    state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
    
    # Clean keys (remove module. and _orig_mod. prefixes)
    cleaned_state = {}
    for k, v in state_dict.items():
        key = k.replace('module.', '').replace('_orig_mod.', '')
        cleaned_state[key] = v
    
    # Load weights
    missing, unexpected = model.load_state_dict(cleaned_state, strict=False)
    if missing:
        print(f"  Missing keys: {len(missing)}")
    if unexpected:
        print(f"  Unexpected keys: {len(unexpected)}")
    
    model = model.to(cfg.DEVICE).eval()
    print(f"Model loaded successfully ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")
    
    # Print configuration
    print("\n" + "="*70)
    print("V3 INFERENCE WITH HYSTERESIS THRESHOLDING")
    print("="*70)
    print(f"Patch size: {cfg.PATCH_SIZE}")
    print(f"Overlap: {cfg.OVERLAP}")
    print(f"Hysteresis: HIGH={cfg.THRESHOLD_HIGH}, LOW={cfg.THRESHOLD_LOW}")
    print(f"Closing: {cfg.USE_CLOSING} (radius={cfg.CLOSING_RADIUS})")
    print(f"Min component: {cfg.MIN_COMPONENT_SIZE}")
    print("="*70)
    
    # Create submission
    with zipfile.ZipFile(cfg.SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for idx, row in test_df.iterrows():
            image_id = row["id"]
            tif_path = cfg.TEST_IMAGES_DIR / f"{image_id}.tif"
            
            print(f"\n[{idx+1}/{len(test_df)}] Processing {image_id}...")
            volume = tifffile.imread(str(tif_path))
            print(f"  Volume shape: {volume.shape}")
            
            # Run inference
            prob_map = sliding_window_inference(
                model=model,
                volume=volume,
                patch_size=cfg.PATCH_SIZE,
                overlap=cfg.OVERLAP,
                device=cfg.DEVICE,
                use_amp=cfg.USE_AMP
            )
            
            # Apply hysteresis post-processing
            prediction = hysteresis_postprocess(prob_map, cfg)
            
            # Save
            out_path = cfg.OUTPUT_DIR / f"{image_id}.tif"
            tifffile.imwrite(out_path, prediction.astype(np.uint8))
            zf.write(out_path, arcname=f"{image_id}.tif")
            out_path.unlink()
            
            # Cleanup
            del volume, prob_map, prediction
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    print("\n" + "="*70)
    print(f"Submission created: {cfg.SUBMISSION_ZIP}")
    print("="*70)


if __name__ == "__main__":
    main()

## Hysteresis Threshold Tuning Guide

### V3 Model Probability Distribution

Your V3 model has unusual output characteristics:
- **Mean probability**: ~0.04 (very low)
- **Max probability**: ~0.99 (but very sparse)
- **FG at threshold 0.35**: only 0.275%

This means standard thresholds (0.5-0.8) won't work. We need MUCH lower thresholds.

### How Hysteresis Thresholding Works

```
Probability Map:     Seeds (HIGH):       Weak Regions (LOW):    Final Result:
   0.20 0.15 0.04      1    1    0          1    1    1            1    1    1
   0.12 0.02 0.08      0    0    0          1    0    1            1    0    1  
   0.10 0.05 0.01      0    0    0          1    1    0            1    1    0
   
Seeds propagate through weak regions that are connected to them.
Isolated weak regions (not connected to any seed) are removed.
```

### Parameter Tuning for V3

| FG% Target | THRESHOLD_HIGH | THRESHOLD_LOW | Notes |
|------------|----------------|---------------|-------|
| ~5% | 0.20 | 0.04 | Conservative |
| ~10% | 0.15 | 0.03 | **Recommended** (matches V2's 10% FG) |
| ~15% | 0.10 | 0.02 | Aggressive |

### Threshold Adjustment Guide

From your test run:
- `Seeds (>= 0.80)`: 0.081% ← Way too few seeds!
- `Weak (>= 0.35)`: 0.275% ← Way too little expansion!
- `Final`: 0.251% ← Need ~10% for good LB score

**New thresholds (current defaults):**
- `THRESHOLD_HIGH = 0.15` → Should give ~2-5% seeds
- `THRESHOLD_LOW = 0.03` → Should give ~15-20% weak regions
- After hysteresis + dust removal → Target ~10% FG

### Quick Tuning Process

1. Run inference and check the printed statistics
2. Adjust thresholds based on:
   - If FG% < 8%: Lower both thresholds
   - If FG% > 12%: Raise both thresholds
   - If too fragmented: Increase CLOSING_RADIUS
   - If too much noise: Increase MIN_COMPONENT_SIZE